# Circuit rewiring example

Adds a new **edge population** to a SONATA circuit: *which* neurons are connected is drawn from a
structural connectivity model per pathway, and every connection (synapse) gets

- a **location on the postsynaptic neuron's morphology** (from a morphology-locations block), and
- **physiological parameters** (from a synaptic model, e.g. Tsodyks-Markram).

This notebook runs entirely locally on a small example circuit: it does not talk to entitycore, and the
input circuit is never modified (it is copied into the output folder first).

Three pathways are defined:

| Pathway | Source | Target | Connectivity model | Synaptic model |
|---|---|---|---|---|
| `Feedforward` | virtual `VPM` | all `S1nonbarrel_neurons` | Erdos-Renyi | excitatory |
| `LocalExc` | excitatory `S1nonbarrel_neurons` | all `S1nonbarrel_neurons` | distance-dependent | excitatory |
| `LocalInh` | inhibitory `S1nonbarrel_neurons` | all `S1nonbarrel_neurons` | Erdos-Renyi | inhibitory |

The two local pathways connect the same pair of node populations, so their synapses end up in the same edge
population; the feedforward pathway gets its own.

In [ ]:
from datetime import datetime
from pathlib import Path

from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask

from obi_one.scientific.blocks.distributions.uniform import IntUniformDistribution
from obi_one.scientific.blocks.morphology_locations.random import RandomMorphologyLocations
from obi_one.scientific.blocks.neuron_sets.population import VirtualPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets.predefined import BiophysicalPopulationPredefinedNeuronSet
from obi_one.scientific.blocks.neuron_sets.specific import AllBiophysicalNeurons
from obi_one.scientific.blocks.structural_conn_models.distance_dependent import DistanceDependentConnModel
from obi_one.scientific.blocks.structural_conn_models.erdos_renyi import ErdosRenyiConnModel
from obi_one.scientific.blocks.synaptic_models.tsodyks_markram.block import (
    ExcitatoryTsodyksMarkramSynapticModel,
    InhibitoryTsodyksMarkramSynapticModel,
)
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.circuit_rewiring import (
    CircuitConnectivityPathway,
    CircuitRewiringScanConfig,
)

In [ ]:
input_root = Path("../../../../data/tiny_circuits")
# A fresh folder per run, so re-running the notebook never collides with earlier output
output_root = Path("../../../../../../obi-output/circuit_rewiring") / datetime.now().strftime("%Y%m%d_%H%M%S")

### Parameterize the task

A `CircuitRewiringScanConfig` is assembled from blocks:

- **Neuron sets** select the neurons on the source side of a pathway (they may be virtual).
- **Connectivity models** decide which source/target pairs are connected (`ErdosRenyiConnModel`,
  `DistanceDependentConnModel`, `SgsgConnModel`).
- **Distributions** give the number of synapses per connection: each connection is replicated that many times.
- **Synaptic models** provide the physiological parameters of each synapse.
- **Morphology locations** place the synapses on the postsynaptic morphology. The `neuron_set` of this block
  defines the **target neurons** of every pathway that uses it, and it must be biophysical. The number of
  locations per neuron is set automatically to the number of synapses it receives.
- A **pathway** ties one source, one connectivity model, one synapse-count distribution, one synaptic model and one
  morphology-locations block together.

In [ ]:
config = CircuitRewiringScanConfig.empty_config()

# Info
config.set(
    Info(campaign_name="Circuit rewiring example", campaign_description="Feedforward and local connectivity"),
    name="info",
)

# Neuron sets. Sources may be virtual; the target neurons come from the morphology locations (below)
vpm = VirtualPopulationNeuronSet(population="VPM")
s1_exc = BiophysicalPopulationPredefinedNeuronSet(population="S1nonbarrel_neurons", node_set="Excitatory")
s1_inh = BiophysicalPopulationPredefinedNeuronSet(population="S1nonbarrel_neurons", node_set="Inhibitory")
s1_all = AllBiophysicalNeurons()
config.add(vpm, name="VPM")
config.add(s1_exc, name="S1Exc")
config.add(s1_inh, name="S1Inh")
config.add(s1_all, name="S1All")

# Structural connectivity models
feedforward_conn = ErdosRenyiConnModel(connection_probability=0.1)
local_exc_conn = DistanceDependentConnModel(scale_factor=0.8, length_scale=150.0)  # length scale in um
local_inh_conn = ErdosRenyiConnModel(connection_probability=0.8)
config.add(feedforward_conn, name="FeedforwardConn")
config.add(local_exc_conn, name="LocalExcConn")
config.add(local_inh_conn, name="LocalInhConn")

# Number of synapses per connection (integers drawn from [low, high))
synapses_feedforward = IntUniformDistribution(low=2, high=6)
synapses_local = IntUniformDistribution(low=1, high=4)
config.add(synapses_feedforward, name="SynapsesFeedforward")
config.add(synapses_local, name="SynapsesLocal")

# Synaptic models (physiological parameters of the synapses)
exc_model = ExcitatoryTsodyksMarkramSynapticModel()
inh_model = InhibitoryTsodyksMarkramSynapticModel()
config.add(exc_model, name="ExcModel")
config.add(inh_model, name="InhModel")

# Locations of the synapses on the postsynaptic dendrites; its neuron set defines the targets
locations = RandomMorphologyLocations(section_types=(3, 4), neuron_set=s1_all.ref)  # 3: basal, 4: apical
config.add(locations, name="DendriticLocations")

# Pathways
config.pathways["Feedforward"] = CircuitConnectivityPathway(
    source_neuron_set=vpm.ref,
    conn_model=feedforward_conn.ref,
    synapses_per_connection=synapses_feedforward.ref,
    synaptic_model=exc_model.ref,
    morphology_locations=locations.ref,
)
config.pathways["LocalExc"] = CircuitConnectivityPathway(
    source_neuron_set=s1_exc.ref,
    conn_model=local_exc_conn.ref,
    synapses_per_connection=synapses_local.ref,
    synaptic_model=exc_model.ref,
    morphology_locations=locations.ref,
)
config.pathways["LocalInh"] = CircuitConnectivityPathway(
    source_neuron_set=s1_inh.ref,
    conn_model=local_inh_conn.ref,
    synapses_per_connection=synapses_local.ref,
    synaptic_model=inh_model.ref,
    morphology_locations=locations.ref,
)

# Initialization: the circuit to rewire (a local circuit here; entitycore IDs are also supported)
circuit_path = input_root / "N_10__top_nodes_dim6" / "circuit_config.json"
config.set(
    CircuitRewiringScanConfig.Initialize(
        circuit=Circuit(name="N_10__top_nodes_dim6", path=str(circuit_path)),
        edge_population_name="rewired",  # prefix of the new edge population name(s)
        seed=0,
    ),
    name="initialize",
)

# Validated config
validated_config = config.validated_config()

In [ ]:
# Generate the (single-instance) scan
grid_scan = GridScanGenerationTask(
    form=validated_config, output_root=str(output_root), coordinate_directory_option="ZERO_INDEX"
)
grid_scan.execute()

### Run the task locally

In [ ]:
run_tasks_for_generated_scan(grid_scan)

### Inspect the result

The rewired circuit is written to `<output_root>/0/SONATA`. The new edge population(s) were appended to its
`circuit_config.json`; the original edge populations are untouched.

In [ ]:
single_config = grid_scan.single_configs[0]
rewired = Circuit(name="rewired", path=str(single_config.coordinate_output_root / "SONATA" / "circuit_config.json"))
original = validated_config.initialize.circuit

original_edge_populations = set(original.sonata_circuit.edges.population_names)
print(f"Original edge populations: {sorted(original_edge_populations)}\n")
for name in rewired.sonata_circuit.edges.population_names:
    edges = rewired.sonata_circuit.edges[name]
    tag = "" if name in original_edge_populations else "  <-- new"
    print(f"{name}: {edges.source.name} -> {edges.target.name}, {edges.size} synapses{tag}")

In [ ]:
new_edge_populations = [
    name for name in rewired.sonata_circuit.edges.population_names if name not in original_edge_populations
]
for name in new_edge_populations:
    edges = rewired.sonata_circuit.edges[name]
    properties = sorted(p for p in edges.property_names if not p.startswith("@"))
    print(f"{name}\n  edge properties: {properties}\n")

In [ ]:
# A few synapses of the first new edge population: connectivity, postsynaptic location and parameters
edges = rewired.sonata_circuit.edges[new_edge_populations[0]]
table = edges.get(edges.ids(), properties=["@source_node", "@target_node", *sorted(p for p in edges.property_names if not p.startswith("@"))])
table.head()

### Check the pathways

The synaptic model of each pathway sets `syn_type_id` (excitatory vs. inhibitory), and the number of synapses per
connection follows the configured distributions.

In [ ]:
for name in new_edge_populations:
    edges = rewired.sonata_circuit.edges[name]
    table = edges.get(edges.ids(), properties=["@source_node", "@target_node", "syn_type_id"])
    n_connections = len(table[["@source_node", "@target_node"]].drop_duplicates())
    synapses_per_connection = table.groupby(["@source_node", "@target_node"]).size()
    print(f"{name}: {len(table)} synapses in {n_connections} connections "
          f"({synapses_per_connection.min()}-{synapses_per_connection.max()} synapses per connection)")
    print("  synapses per syn_type_id:", table["syn_type_id"].value_counts().to_dict())